In [ ]:
#import libraries

import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

In [ ]:
#dataset path

dataset_path = "desktop/nna272/msc_thesis/breast_cancer/EMBED_dataset"
metadata_path = os.path.join(dataset_path, "metadata.csv")

In [ ]:
#load metadata

df = pd.read_csv(metadata_path)

print(df.head())
print(df.shape)

In [ ]:
#deleting null values

df = df.dropna()

print(df.isnull().sum())

In [ ]:
#removing duplicate rows

df = df.drop_duplicates()

print(df.duplicated().sum())

In [ ]:
#removing duplicate rows

df = df.drop_duplicates()

print(df.duplicated().sum())

In [ ]:
#checking class distribution

print(df['pathology'].value_counts())

In [ ]:
#encoding labels

label_map = {
    'BENIGN':0,
    'MALIGNANT':1,
    'NORMAL':2
}

df['label'] = df['pathology'].map(label_map)

In [ ]:
#creating image paths

df['image_path'] = df['image_file_path'].apply(
    lambda x: os.path.join(dataset_path, x)
)

print(df['image_path'].head())

In [ ]:
#resizing and preprocessing images

IMG_SIZE = 224

images = []
labels = []

for idx, row in tqdm(df.iterrows(), total=len(df)):

    img_path = row['image_path']

    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

    if img is not None:

        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

        img = img / 255.0

        images.append(img)

        labels.append(row['label'])

In [ ]:
#adding channel dimension

X = X.reshape(-1, 224, 224, 1)

print(X.shape)

In [ ]:
#train validation test split

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

In [ ]:
#saving preprocessed arrays

np.save("X_train.npy", X_train)
np.save("X_val.npy", X_val)
np.save("X_test.npy", X_test)

np.save("y_train.npy", y_train)
np.save("y_val.npy", y_val)
np.save("y_test.npy", y_test)

In [ ]:
#checking sample image

import matplotlib.pyplot as plt

plt.imshow(X_train[0].reshape(224,224), cmap='gray')
plt.title(np.argmax(y_train[0]))
plt.axis('off')
plt.show()

In [ ]:
#loading images and masks

image_files = sorted(os.listdir(dataset_path))
mask_files = sorted(os.listdir(mask_path))

images = []
masks = []

for img_file, mask_file in tqdm(zip(image_files, mask_files), total=len(image_files)):

    img = cv2.imread(os.path.join(dataset_path, img_file), cv2.IMREAD_GRAYSCALE)

    mask = cv2.imread(os.path.join(mask_path, mask_file), cv2.IMREAD_GRAYSCALE)

    if img is not None and mask is not None:

        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

        mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE))

        img = img / 255.0

        mask = mask / 255.0

        mask = (mask > 0.5).astype(np.float32)

        images.append(img)

        masks.append(mask)

X = np.array(images)
y = np.array(masks)

X = X.reshape(-1, IMG_SIZE, IMG_SIZE, 1)
y = y.reshape(-1, IMG_SIZE, IMG_SIZE, 1)

print(X.shape)
print(y.shape)

In [ ]:
#train validation split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print(X_train.shape)
print(X_val.shape)

In [ ]:
#unet model

inputs = Input((IMG_SIZE, IMG_SIZE, 1))

c1 = Conv2D(64, 3, activation='relu', padding='same')(inputs)
c1 = Conv2D(64, 3, activation='relu', padding='same')(c1)
p1 = MaxPooling2D((2,2))(c1)

c2 = Conv2D(128, 3, activation='relu', padding='same')(p1)
c2 = Conv2D(128, 3, activation='relu', padding='same')(c2)
p2 = MaxPooling2D((2,2))(c2)

c3 = Conv2D(256, 3, activation='relu', padding='same')(p2)
c3 = Conv2D(256, 3, activation='relu', padding='same')(c3)
p3 = MaxPooling2D((2,2))(c3)

c4 = Conv2D(512, 3, activation='relu', padding='same')(p3)
c4 = Conv2D(512, 3, activation='relu', padding='same')(c4)

u5 = UpSampling2D((2,2))(c4)
u5 = concatenate([u5, c3])

c5 = Conv2D(256, 3, activation='relu', padding='same')(u5)
c5 = Conv2D(256, 3, activation='relu', padding='same')(c5)

u6 = UpSampling2D((2,2))(c5)
u6 = concatenate([u6, c2])

c6 = Conv2D(128, 3, activation='relu', padding='same')(u6)
c6 = Conv2D(128, 3, activation='relu', padding='same')(c6)

u7 = UpSampling2D((2,2))(c6)
u7 = concatenate([u7, c1])

c7 = Conv2D(64, 3, activation='relu', padding='same')(u7)
c7 = Conv2D(64, 3, activation='relu', padding='same')(c7)

outputs = Conv2D(1, 1, activation='sigmoid')(c7)

model = Model(inputs=[inputs], outputs=[outputs])

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
#training model

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=8,
    callbacks=[early_stop]
)

In [ ]:
#saving model

model.save("embed_segmentation_unet.h5")

In [ ]:
#visualizing predictions

preds = model.predict(X_val[:5])

plt.figure(figsize=(15,10))

for i in range(5):

    plt.subplot(5,3,i*3+1)
    plt.imshow(X_val[i].reshape(IMG_SIZE, IMG_SIZE), cmap='gray')
    plt.title("Image")
    plt.axis('off')

    plt.subplot(5,3,i*3+2)
    plt.imshow(y_val[i].reshape(IMG_SIZE, IMG_SIZE), cmap='gray')
    plt.title("True Mask")
    plt.axis('off')

    plt.subplot(5,3,i*3+3)
    plt.imshow(preds[i].reshape(IMG_SIZE, IMG_SIZE) > 0.5, cmap='gray')
    plt.title("Predicted Mask")
    plt.axis('off')

plt.tight_layout()
plt.show()